# 总体比例的假设检验完整教程：从正态近似到精确法

## 📚 教学目标
1. 掌握比例检验的假设设定和条件检查
2. 理解正态近似法的条件（np ≥ 5, nq ≥ 5）
3. 计算检验统计量 z 和 p 值，完成三种方法的判断
4. 用 scipy.stats 进行比例检验并验证手算结果
5. 理解精确法（二项分布）与近似法（正态分布）的区别

**参考书**：《基础统计学(第14版)》（Triola）第 8-2 节
**教学策略**：先用正态近似法理解核心逻辑，再学习精确法处理小样本情况

---

## 1. 场景设定：比例检验的实际应用

### 🎯 两个经典问题

**问题1（右侧检验）**：调查显示 926 位互联网用户中有 52% 使用双重认证。能否认为"大多数"（>50%）用户使用双重认证？

**问题2（左侧检验）**：19,136 名成年人中有 29.2% 曾经梦游。能否认为"少于 30%"的成年人有过梦游？

### 📖 书中的定义

> **总体比例的假设检验**用于检验关于总体比例 p、概率或百分比的命题。
> 当样本量足够大时，可以用正态分布近似二项分布。

### 📐 比例检验的核心公式

$$z = \frac{\hat{p} - p}{\sqrt{\frac{pq}{n}}}$$

其中：
- $\hat{p} = x/n$ = 样本比例
- p = 原假设中的总体比例
- q = 1 - p

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import matplotlib.font_manager as fm
_cn_fonts = [f.name for f in fm.fontManager.ttflist if any(kw in f.name for kw in ['Hei', 'Song', 'PingFang', 'Arial Unicode', 'Noto Sans CJK', 'SimHei', 'Microsoft YaHei'])]
plt.rcParams['font.sans-serif'] = _cn_fonts[:3] + ['DejaVu Sans'] if _cn_fonts else ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 比例检验的条件

### 📐 三个必要条件

使用正态近似法进行比例检验，必须满足以下三个条件：

| 条件 | 说明 |
|------|------|
| 条件 1 | 样本为简单随机样本 |
| 条件 2 | 满足二项分布条件：固定试验次数、独立、两种结果、概率不变 |
| 条件 3 | **np ≥ 5 且 nq ≥ 5**（p 来自 H₀，不是样本比例） |

### ⚠️ 重要提醒

条件 3 中的 p 是**原假设中的比例**，不是样本比例 $\hat{p}$！

### 📝 不满足条件时的替代方法

| 方法 | 适用情况 |
|------|----------|
| 精确法（二项分布） | np < 5 或 nq < 5 |
| 自助重采样法 | 小样本 |
| 置换检验 | 非参数方法 |
| 符号检验 | 13-2 节 |

In [ ]:
# ========== 条件检查示例 ==========

print("=" * 60)
print("📋 条件检查示例")
print("=" * 60)

# 示例1: 互联网双重认证
n1, p_hat1, p0_1 = 926, 0.52, 0.50
print(f"\n📊 示例1: 互联网双重认证")
print(f"  n = {n1}, p̂ = {p_hat1}, H₀: p = {p0_1}")
print(f"  np₀ = {n1 * p0_1:.0f} ≥ 5? {'✓' if n1 * p0_1 >= 5 else '✗'}")
print(f"  nq₀ = {n1 * (1-p0_1):.0f} ≥ 5? {'✓' if n1 * (1-p0_1) >= 5 else '✗'}")
print(f"  结论: {'条件满足，可用正态近似' if n1*p0_1>=5 and n1*(1-p0_1)>=5 else '条件不满足'}")

# 示例2: 梦游比例
n2, p_hat2, p0_2 = 19136, 0.292, 0.30
print(f"\n📊 示例2: 梦游比例")
print(f"  n = {n2}, p̂ = {p_hat2}, H₀: p = {p0_2}")
print(f"  np₀ = {n2 * p0_2:.0f} ≥ 5? {'✓' if n2 * p0_2 >= 5 else '✗'}")
print(f"  nq₀ = {n2 * (1-p0_2):.0f} ≥ 5? {'✓' if n2 * (1-p0_2) >= 5 else '✗'}")
print(f"  结论: {'条件满足，可用正态近似' if n2*p0_2>=5 and n2*(1-p0_2)>=5 else '条件不满足'}")

# 示例3: 小样本
n3, p_hat3, p0_3 = 20, 0.30, 0.50
print(f"\n📊 示例3: 小样本")
print(f"  n = {n3}, p̂ = {p_hat3}, H₀: p = {p0_3}")
print(f"  np₀ = {n3 * p0_3:.0f} ≥ 5? {'✓' if n3 * p0_3 >= 5 else '✗'}")
print(f"  nq₀ = {n3 * (1-p0_3):.0f} ≥ 5? {'✓' if n3 * (1-p0_3) >= 5 else '✗'}")
print(f"  结论: {'条件满足' if n3*p0_3>=5 and n3*(1-p0_3)>=5 else '条件不满足，需用精确法'}")

---

## 3. 左侧检验完整示例：梦游比例

### 🎯 问题

19,136 名成年人中有 29.2%（5,588 人）曾经梦游。能否认为"少于 30%"的成年人有过梦游？（α = 0.05）

### 📐 假设设定

$H_0: p = 0.30$
$H_1: p < 0.30$ （原命题，左侧检验）

### 📐 检验统计量

$$z = \frac{\hat{p} - p}{\sqrt{\frac{pq}{n}}} = \frac{0.292 - 0.30}{\sqrt{\frac{(0.30)(0.70)}{19136}}} = -2.41$$

In [ ]:
# ========== 左侧检验: 梦游比例 ==========

print("=" * 60)
print("📋 梦游比例 - 左侧检验")
print("=" * 60)

# 数据
n = 19136
x = 5588
p_hat = x / n
p_0 = 0.30
q_0 = 1 - p_0
alpha = 0.05

print(f"\n📊 步骤 1-3: 假设设定")
print(f"  H₀: p = {p_0}")
print(f"  H₁: p < {p_0} (原命题)")
print(f"  检验类型: 左侧检验")

print(f"\n📊 步骤 4: 显著性水平 α = {alpha}")

print(f"\n📊 步骤 5: 检查条件")
print(f"  样本量 n = {n:,}")
print(f"  成功次数 x = {x:,}")
print(f"  样本比例 p̂ = {p_hat:.4f}")
print(f"  np₀ = {n * p_0:.0f} ≥ 5 ✓")
print(f"  nq₀ = {n * q_0:.0f} ≥ 5 ✓")

# 计算检验统计量
z_stat = (p_hat - p_0) / np.sqrt(p_0 * q_0 / n)
print(f"\n📊 步骤 6: 计算检验统计量")
print(f"  z = (p̂ - p) / √(pq/n)")
print(f"  z = ({p_hat:.4f} - {p_0}) / √({p_0}×{q_0}/{n:,})")
print(f"  z = {z_stat:.4f}")

# 计算 p 值 (左侧检验)
p_value = stats.norm.cdf(z_stat)
print(f"\n📊 步骤 6 (续): 计算 p 值")
print(f"  左侧检验: p 值 = P(Z < {z_stat:.2f})")
print(f"  p 值 = {p_value:.4f}")

# 临界值
z_critical = stats.norm.ppf(alpha)
print(f"\n📊 步骤 6 (续): 临界值法")
print(f"  临界值 -z_α = {z_critical:.3f}")
print(f"  检验统计量 z = {z_stat:.4f}")

# 判断
print(f"\n📊 步骤 7: 做出判断")
print(f"  p 值法: p = {p_value:.4f} {'< α → 拒绝 H₀' if p_value < alpha else '≥ α → 不能拒绝 H₀'}")
print(f"  临界值法: z = {z_stat:.4f} {'< -z_α → 拒绝 H₀' if z_stat < z_critical else '≥ -z_α → 不能拒绝 H₀'}")

# 结论
print(f"\n📊 步骤 8: 结论")
if p_value < alpha:
    print(f"  有足够的证据支持'少于 30% 的成年人有过梦游'")
else:
    print(f"  没有足够的证据支持'少于 30% 的成年人有过梦游'")

print("\n" + "=" * 60)

---

## 4. 右侧检验完整示例：互联网双重认证

### 🎯 问题

926 位互联网用户中有 52% 使用双重认证。能否认为"大多数"（>50%）用户使用双重认证？（α = 0.05）

### 📐 假设设定

$H_0: p = 0.50$
$H_1: p > 0.50$ （原命题，右侧检验）

In [ ]:
# ========== 右侧检验: 互联网双重认证 ==========

print("=" * 60)
print("📋 互联网双重认证 - 右侧检验")
print("=" * 60)

# 数据
n = 926
p_hat = 0.52
p_0 = 0.50
q_0 = 1 - p_0
alpha = 0.05

print(f"\n📊 步骤 1-3: 假设设定")
print(f"  H₀: p = {p_0}")
print(f"  H₁: p > {p_0} (原命题)")
print(f"  检验类型: 右侧检验")

print(f"\n📊 步骤 4: 显著性水平 α = {alpha}")

print(f"\n📊 步骤 5: 检查条件")
print(f"  样本量 n = {n}")
print(f"  样本比例 p̂ = {p_hat}")
print(f"  np₀ = {n * p_0:.0f} ≥ 5 ✓")
print(f"  nq₀ = {n * q_0:.0f} ≥ 5 ✓")

# 计算检验统计量
z_stat = (p_hat - p_0) / np.sqrt(p_0 * q_0 / n)
print(f"\n📊 步骤 6: 计算检验统计量")
print(f"  z = (p̂ - p) / √(pq/n)")
print(f"  z = ({p_hat} - {p_0}) / √({p_0}×{q_0}/{n})")
print(f"  z = {z_stat:.4f}")

# 计算 p 值 (右侧检验)
p_value = 1 - stats.norm.cdf(z_stat)
print(f"\n📊 步骤 6 (续): 计算 p 值")
print(f"  右侧检验: p 值 = P(Z > {z_stat:.2f})")
print(f"  p 值 = {p_value:.4f}")

# 临界值
z_critical = stats.norm.ppf(1 - alpha)
print(f"\n📊 步骤 6 (续): 临界值法")
print(f"  临界值 z_α = {z_critical:.3f}")
print(f"  检验统计量 z = {z_stat:.4f}")

# 判断
print(f"\n📊 步骤 7: 做出判断")
print(f"  p 值法: p = {p_value:.4f} {'< α → 拒绝 H₀' if p_value < alpha else '≥ α → 不能拒绝 H₀'}")
print(f"  临界值法: z = {z_stat:.4f} {'> z_α → 拒绝 H₀' if z_stat > z_critical else '≤ z_α → 不能拒绝 H₀'}")

# 结论
print(f"\n📊 步骤 8: 结论")
if p_value < alpha:
    print(f"  有足够的证据支持'大多数互联网用户使用双重认证'")
else:
    print(f"  没有足够的样本证据支持'大多数互联网用户使用双重认证'")

print("\n" + "=" * 60)

---

## 5. 三种检验方法对比

### 📐 方法对比

进行假设检验有三种等价方法（对于比例检验，置信区间法可能给出不同结论）：

| 方法 | 判断标准 | 优点 |
|------|---------|------|
| p 值法 | p < α → 拒绝 H₀ | 提供精确的证据强度 |
| 临界值法 | 统计量在拒绝域内 → 拒绝 H₀ | 直观，易于理解 |
| 置信区间法 | 假设值不在 CI 内 → 拒绝 H₀ | 提供参数估计范围 |

### ⚠️ 重要提醒

对于**比例检验**，置信区间法与前两种方法可能给出不同结论！
- p 值法和临界值法使用 H₀ 中比例的标准差
- 置信区间法使用样本比例的标准差

建议：**用 p 值法或临界值法做假设检验，用置信区间法做参数估计**。

In [ ]:
# ========== 三种方法对比 ==========

print("=" * 60)
print("📋 三种方法对比: 梦游比例例子")
print("=" * 60)

n = 19136
p_hat = 0.292
p_0 = 0.30
alpha = 0.05

# 方法1: p 值法
z_stat = (p_hat - p_0) / np.sqrt(p_0 * (1-p_0) / n)
p_value = stats.norm.cdf(z_stat)
print(f"\n📊 方法1: p 值法")
print(f"  z = {z_stat:.4f}")
print(f"  p 值 = {p_value:.6f}")
print(f"  判断: p = {p_value:.4f} {'< α' if p_value < alpha else '≥ α'} → {'拒绝 H₀' if p_value < alpha else '不能拒绝 H₀'}")

# 方法2: 临界值法
z_critical = stats.norm.ppf(alpha)
print(f"\n📊 方法2: 临界值法")
print(f"  临界值 -z_α = {z_critical:.3f}")
print(f"  检验统计量 z = {z_stat:.4f}")
print(f"  判断: z = {z_stat:.4f} {'< -z_α' if z_stat < z_critical else '≥ -z_α'} → {'拒绝 H₀' if z_stat < z_critical else '不能拒绝 H₀'}")

# 方法3: 置信区间法
confidence_level = 1 - alpha  # 单侧检验对应 90% CI
z_ci = stats.norm.ppf(1 - alpha/2)  # 双侧临界值
se = np.sqrt(p_hat * (1-p_hat) / n)  # 使用样本比例的标准差
ci_lower = p_hat - z_ci * se
ci_upper = p_hat + z_ci * se
print(f"\n📊 方法3: 置信区间法")
print(f"  90% 置信区间: ({ci_lower:.4f}, {ci_upper:.4f})")
print(f"  H₀ 假设值: {p_0}")
print(f"  判断: {p_0} {'不在' if p_0 < ci_lower or p_0 > ci_upper else '在'} CI 内 → {'拒绝 H₀' if p_0 < ci_lower or p_0 > ci_upper else '不能拒绝 H₀'}")

print(f"\n💡 结论: 三种方法的判断结果{'相同' if (p_value<alpha) == (z_stat<z_critical) == (p_0<ci_lower or p_0>ci_upper) else '可能不同'}")

print("\n" + "=" * 60)

---

## 6. 用 scipy.stats 验证

### 🔬 scipy 中的比例检验

scipy 提供了 `stats.binomtest`（精确法）来进行比例检验。

注意：scipy 没有直接的"正态近似法"函数，但我们可以用 `stats.binomtest` 或手动计算验证。

In [ ]:
# ========== scipy 验证 ==========

print("=" * 60)
print("📋 scipy 验证")
print("=" * 60)

# 梦游比例 (左侧检验)
print("\n📊 示例1: 梦游比例 (左侧检验)")
n1, x1, p0_1 = 19136, 5588, 0.30
p_hat1 = x1 / n1

# 手算 (正态近似)
z_manual = (p_hat1 - p0_1) / np.sqrt(p0_1 * (1-p0_1) / n1)
p_manual = stats.norm.cdf(z_manual)

# scipy 精确法
result1 = stats.binomtest(x1, n1, p0_1, alternative='less')
p_scipy1 = result1.pvalue

print(f"  手算 z = {z_manual:.6f}")
print(f"  手算 p (正态近似) = {p_manual:.6f}")
print(f"  scipy p (精确法) = {p_scipy1:.6f}")
print(f"  ✅ 两种方法结果接近！")

# 互联网双重认证 (右侧检验)
print("\n📊 示例2: 互联网双重认证 (右侧检验)")
n2, p_hat2, p0_2 = 926, 0.52, 0.50
x2 = round(n2 * p_hat2)

# 手算 (正态近似)
z_manual2 = (p_hat2 - p0_2) / np.sqrt(p0_2 * (1-p0_2) / n2)
p_manual2 = 1 - stats.norm.cdf(z_manual2)

# scipy 精确法
result2 = stats.binomtest(x2, n2, p0_2, alternative='greater')
p_scipy2 = result2.pvalue

print(f"  手算 z = {z_manual2:.6f}")
print(f"  手算 p (正态近似) = {p_manual2:.6f}")
print(f"  scipy p (精确法) = {p_scipy2:.6f}")
print(f"  ✅ 两种方法结果接近！")

print("\n" + "=" * 60)

---

## 7. 精确法：二项分布直接计算

### 📐 精确法原理

当样本量较小（np < 5 或 nq < 5）时，正态近似法不再适用。此时使用**精确法**，直接基于二项分布计算 p 值。

**精确法的 p 值计算**：

| 检验类型 | p 值 |
|---------|------|
| 左侧检验 | P(X ≤ x) |
| 右侧检验 | P(X ≥ x) |
| 双侧检验 | 2 × min(左侧, 右侧) |

### 📐 连续性校正

精确法的一个改进是**连续性校正**：
- 左侧检验: p = P(X ≤ x) - ½P(X = x)
- 右侧检验: p = P(X ≥ x) - ½P(X = x)

In [ ]:
# ========== 精确法示例 ==========

print("=" * 60)
print("📋 精确法 vs 正态近似法")
print("=" * 60)

# 小样本示例
n = 20
x = 8
p_0 = 0.5
alpha = 0.05

print(f"\n📊 小样本示例:")
print(f"  n = {n}, x = {x}, H₀: p = {p_0}")
print(f"  np₀ = {n*p_0:.0f}, nq₀ = {n*(1-p_0):.0f}")

# 精确法 (二项分布)
result_exact = stats.binomtest(x, n, p_0, alternative='two-sided')
p_exact = result_exact.pvalue
print(f"\n📊 精确法 (二项分布):")
print(f"  p 值 = {p_exact:.6f}")

# 正态近似法
p_hat = x / n
z_approx = (p_hat - p_0) / np.sqrt(p_0 * (1-p_0) / n)
p_approx = 2 * (1 - stats.norm.cdf(abs(z_approx)))
print(f"\n📊 正态近似法:")
print(f"  z = {z_approx:.4f}")
print(f"  p 值 = {p_approx:.6f}")

# 比较
print(f"\n📊 比较:")
print(f"  精确法 p = {p_exact:.6f}")
print(f"  近似法 p = {p_approx:.6f}")
print(f"  差异 = {abs(p_exact - p_approx):.6f}")
print(f"  💡 小样本时两种方法可能有差异，精确法更可靠")

print("\n" + "=" * 60)

---

## 8. 可视化：检验结果

### 📊 图示说明

通过可视化可以直观理解假设检验的判断过程：
- 蓝色曲线：标准正态分布
- 红色区域：拒绝域
- 绿色区域：接受域
- 橙色线：检验统计量的位置

In [ ]:
# ========== 可视化两种检验 ==========

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左图: 左侧检验 (梦游)
ax = axes[0]
x = np.linspace(-4, 4, 1000)
y = stats.norm.pdf(x)
ax.plot(x, y, 'b-', linewidth=2, label='Standard Normal')
ax.fill_between(x, 0, y, alpha=0.1, color='steelblue')

z_stat_left = -2.41
z_crit_left = stats.norm.ppf(0.05)

x_reject = np.linspace(-4, z_crit_left, 500)
ax.fill_between(x_reject, 0, stats.norm.pdf(x_reject), alpha=0.4, color='#e74c3c',
                label=f'Rejection Region (α=0.05)')
x_accept = np.linspace(z_crit_left, 4, 500)
ax.fill_between(x_accept, 0, stats.norm.pdf(x_accept), alpha=0.2, color='#2ecc71',
                label='Acceptance Region')
ax.axvline(x=z_crit_left, color='red', linestyle='--', linewidth=2,
           label=f'Critical Value z={z_crit_left:.3f}')
ax.axvline(x=z_stat_left, color='#e67e22', linestyle='-', linewidth=2.5,
           label=f'Test Statistic z={z_stat_left:.2f}')

ax.set_xlabel('Z Value', fontsize=12)
ax.set_ylabel('Probability Density', fontsize=12)
ax.set_title('Left-Tailed Test: Sleepwalking (p < 0.30)', fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='upper left')
ax.grid(alpha=0.3)

# 右图: 右侧检验 (双重认证)
ax = axes[1]
ax.plot(x, y, 'b-', linewidth=2, label='Standard Normal')
ax.fill_between(x, 0, y, alpha=0.1, color='steelblue')

z_stat_right = 1.25
z_crit_right = stats.norm.ppf(0.95)

x_reject = np.linspace(z_crit_right, 4, 500)
ax.fill_between(x_reject, 0, stats.norm.pdf(x_reject), alpha=0.4, color='#e74c3c',
                label=f'Rejection Region (α=0.05)')
x_accept = np.linspace(-4, z_crit_right, 500)
ax.fill_between(x_accept, 0, stats.norm.pdf(x_accept), alpha=0.2, color='#2ecc71',
                label='Acceptance Region')
ax.axvline(x=z_crit_right, color='red', linestyle='--', linewidth=2,
           label=f'Critical Value z={z_crit_right:.3f}')
ax.axvline(x=z_stat_right, color='#e67e22', linestyle='-', linewidth=2.5,
           label=f'Test Statistic z={z_stat_right:.2f}')

ax.set_xlabel('Z Value', fontsize=12)
ax.set_ylabel('Probability Density', fontsize=12)
ax.set_title('Right-Tailed Test: 2FA Usage (p > 0.50)', fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='upper left')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 图解说明：")
print("  左图 (梦游): z = -2.41 落在拒绝域内 → 拒绝 H₀")
print("  右图 (双重认证): z = 1.25 未落在拒绝域内 → 不能拒绝 H₀")
print("  红色区域 = 拒绝域 (α = 0.05)")
print("  绿色区域 = 接受域")
print("  橙色线 = 检验统计量的实际位置")

---

## 📌 核心概念回顾

### 📌 比例检验的条件
- **定义**: 简单随机样本 + 二项分布条件 + np₀ ≥ 5 且 nq₀ ≥ 5
- **公式**: $z = \frac{\hat{p} - p}{\sqrt{pq/n}}$
- **含义**: 衡量样本比例与假设比例之间的标准差距离

### 📌 三种检验方法
- **p 值法**: p < α → 拒绝 H₀
- **临界值法**: 统计量在拒绝域内 → 拒绝 H₀
- **置信区间法**: 假设值不在 CI 内 → 拒绝 H₀
- **注意**: 对于比例检验，CI 法可能给出不同结论

### 📌 精确法 vs 近似法
- **近似法**: 用正态分布近似二项分布，需要大样本
- **精确法**: 直接用二项分布计算，适用于小样本
- **选择**: np₀ ≥ 5 且 nq₀ ≥ 5 用近似法，否则用精确法

### 🔗 完整流程
```
设定假设 → 检查条件 → 计算 z → 求 p 值 → 判断 → 结论
    ↓          ↓         ↓        ↓       ↓       ↓
  H₀/H₁    np≥5?     公式    查表/软件  p<α?   非技术语言
```

### 📝 方法选择指南

| 情况 | 推荐方法 |
|------|----------|
| 大样本 (np≥5, nq≥5) | 正态近似法 |
| 小样本 (np<5 或 nq<5) | 精确法 (二项分布) |
| 非正态总体 | 自助法或置换检验 |

---

## ❌ 常见误区

### ❌ 误区 1: 用样本比例 p̂ 代替假设比例 p 来检查条件
**✓ 正确做法**: 条件 np ≥ 5 和 nq ≥ 5 中的 p 必须是 H₀ 中的比例，不是样本比例。样本比例可能与假设比例不同，用错会导致错误判断。

### ❌ 误区 2: 混淆单侧检验和双侧检验
**✓ 正确做法**: 根据原命题确定检验类型。命题含 ">" 或 "<" 用单侧，含 "≠" 用双侧。双侧检验的 p 值是单侧的两倍。

### ❌ 误区 3: "不能拒绝 H₀" 意味着 "H₀ 为真"
**✓ 正确做法**: 不能拒绝只表示证据不足，不等于证明。样本量太小也可能导致无法拒绝。

### ❌ 误区 4: p 值越小，效应越大
**✓ 正确做法**: p 值反映的是证据的强度，不是效应的大小。大样本可以使微小差异变得"统计显著"。要结合效应量和置信区间来解释。

### ❌ 误区 5: 置信区间法总是与 p 值法一致
**✓ 正确做法**: 对于比例检验，两种方法可能给出不同结论。这是因为它们使用了不同的标准差计算方式。建议用 p 值法做检验，用置信区间法做估计。